In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — VALIDATION SUR UN MATCH JAMAIS VU : WARSAGE (ou Baelen) — 20/11/2026
# Configuration de PRODUCTION : RF-DETR Medium (seul détecteur) + SCOUTIA_PROFIL_KO=2026-11 (détection des buts validée
# à 0 $ sur 11 matchs) + rapport option B ; puis, si JOUEURS n'est pas vide, montage de chaque joueur par le MODULE DE
# PRODUCTION analysis/montage_joueur.py (plus aucun code de montage dans cette cellule).
# Vérité notée AVANT le test : Warsage 1-1, buts 42:06 (vert) et 111:01 (rouge) ; bornes KO1 2:25, fin 1re MT 48:00,
# KO2 63:01, fin 111:51. Baelen 4-3 : voir NOUVEAUX ci-dessous.
# ENTRÉES : la vidéo du match, codeprojectai (avec parseq_scoutia_maillots_v2.pt), mziparameter (NOUVELLE clé Gemini).
# GPU T4 ×2 OBLIGATOIRE (sur processeur, Warsage dépasserait la limite de 12 h de Kaggle). Internet activé.
# GitHub : tous les fichiers à jour (dossier MISE_EN_PROD_19_11). « Save & Run All ». ≈ 1 h 45 d'analyse + ≈ 35-45 min
# par joueur ; Gemini ≈ 2,5 à 4 $ pour l'analyse (estimation) + 0,1 à 0,3 $ par joueur.
# À GARDER (sortie du notebook) : cache_<match>_reference.zip, bout_en_bout_<match>.json, gemini_cache_<match>.jsonl,
# lectures_v30_<match>_v2.json (dans les dossiers joueur_*), rapports PDF, montages.
# ════════════════════════════════════════════════════════════════════════════════════════════
# JOUEURS à monter après l'analyse (vide = analyse seule). Couleur = couleur de son ÉQUIPE ; gardien : couleur de SON maillot.
JOUEURS = []   # ex. [dict(numero=10, couleur="vert"), dict(numero=1, couleur="rouge", gardien=True, couleur_maillot="jaune")]
import os, sys, shutil, logging, builtins, time

os.environ["DEBUG"] = "false"
os.environ["DISABLE_OCR"] = "1"
os.makedirs("/kaggle/working/outputs", exist_ok=True)

MATCH = "baelen"   # ← warsage ou baelen (nouveaux matchs de validation)
# 18/11/2026 : NOMS DES ÉQUIPES pour le rapport (facultatif). Reliés aux tenues détectées par leur COULEUR ; sans
# couleur ou si le lien est ambigu, le rapport affiche les couleurs (« Rouge et blanc », « Bleu »).
EQUIPES = []   # ex. [{"nom": "SRU Verviers", "couleur": "rouge et blanc"}, {"nom": "Adversaire", "couleur": "bleu"}] : spa, franchimont, andrimont, p1_minerois, goe, stembert, wanze, juprelle,
                         #   raeren, minerois_ster (ajouter sa vidéo en entrée du notebook)
MONTAGE_JOUEUR = False   # 15/11 : False = analyse du match seule (test des buts et des équipes, ≈ 35 min de GPU en moins)
T0_BOUT_EN_BOUT = time.time()
import datetime as _dt_prod
_T_DEBUT_GEMINI = time.time()
SEGMENTS = True
SUIVI_SEULEMENT = False
REPLAY = False            # TEST GLOBAL : run complet, nouveau cache de suivi et cache Gemini à jour
KO_DEDUP = "fenetre"
CACHE_REPLAY = ""
SECOND_REGARD = True
OCCASIONS_KO = True
VOIE_BALLON = False
REFLEXION_REDUITE = True
LECTURE_MAILLOTS = False
KICKOFF_AUTO = True       # comme pour un client : coup d'envoi détecté par le pipeline

V_ = "/kaggle/input/datasets/michaelziant"
# 15/11 : les 10 matchs de référence (mêmes données que test_resolution_gemini_cellule.py). Vidéo à ajouter en entrée.
VERITE_EQUIPES = {
 "raeren": {"16:39": "A", "20:08": "A", "27:18": "A", "33:24": "A", "45:25": "A", "46:50": "A", "50:17": "B", "69:12": "B", "91:00": "A"},
 "andrimont": {"06:23": "B", "30:31": "A", "42:34": "A", "48:58": "B", "59:42": "A", "84:06": "A"},
 "franchimont": {"32:33": "A", "49:51": "B", "65:29": "B", "79:42": "A", "99:29": "A"},
 "p1_minerois": {"16:58": "A", "76:55": "A", "110:53": "B"}, "goe": {"83:45": "A", "114:29": "A"},
 "stembert": {"20:27": "A", "76:25": "B", "110:45": "B"}, "wanze": {"22:50": "B", "65:46": "A"}, "juprelle": {"22:50": "A"},
 "minerois_ster": {"55:42": "A", "87:16": "B", "99:02": "A", "103:02": "B", "110:08": "B"},
 "spa": {"19:50": "B", "20:48": "A", "39:06": "A", "49:20": "B", "64:10": "A", "75:03": "A", "89:05": "B", "101:41": "A", "112:45": "A"}}   # 15/11 : tolérance de comparaison 20 → 25 s (heure d'un penalty estimée depuis le coup d'envoi) ; + penalties 19:50 (rouge) et 39:06 (bleu), absents de Veo — score final 3-6
VRAIES_TENUES_TOUTES = {
 "raeren": (("yellow", "black"), ("red", "white")), "andrimont": (("yellow", "blue"), ("red", "white")),
 "franchimont": (("red", "white"), ("green", "black")), "p1_minerois": (("red", "red"), ("white", "black")),
 "goe": (("red", "red"), ("white", "black")), "stembert": (("red", "red"), ("white", "black")),
 "wanze": (("dark blue and red", "dark blue"), ("red and white", "white")), "juprelle": (("blue", "black"), ("red", "white")),
 "minerois_ster": (("red", "white"), ("light blue", "light blue")), "spa": (("blue and black", "black"), ("red", "white"))}
_VID = {"raeren": "raeren-complet/ster-b-raeren-b-2026-04-03.mp4", "andrimont": "stertest/match-p4-andrimont-2026-04-26.mp4",
        "franchimont": "franchimont-complet/ster-b-franchimont-b-2026-04-09.mp4", "p1_minerois": "p1minerois-complet/p1-minerois-2026-03-28.mp4",
        "goe": "goe-complet/ster-b-goe-2026-03-20.mp4", "stembert": "stembert-complet/ster-b-stembert-2026-03-07.mp4",
        "wanze": "wanze-complet/ster-wanze-bas-oha-2026-02-28.mp4", "juprelle": "juprelle-complet/ster-juprelle-2026-01-25.mp4",
        "minerois_ster": "mineroisster-complet/minerois-ster-2025-11-09.mp4", "spa": "spamatchp4/match-p4-2026-08-29.mp4"}
_KO_CONNU = {"raeren": 409.0, "andrimont": 310.0, "minerois_ster": 722.0}
# 17/11 : NOUVEAUX MATCHS DE VALIDATION (buts et bornes notés par l'utilisateur avant le test ; équipes non notées)
NOUVEAUX = {
 # tenues : (maillot, short) en anglais, comme les décrit Gemini ; « » = non noté. Équipe A / B : celles des buts.
 "sruverviers": {"video": "sruverviers/p4-sru-verviers-b-2026-10-10.mp4", "score": "9-0",
                 "buts": {"11:48": "A", "25:33": "A", "55:19": "A", "73:23": "A", "76:27": "A", "83:46": "A",
                          "89:52": "A", "95:52": "A", "112:42": "A"},
                 "tenues": (("red white", ""), ("", "")),          # tous les buts : rouge et blanc ; adversaire non noté
                 "bornes": {"KO1": "11:30", "Fin1MT": "56:45", "KO2": "68:08", "FinMatch": "113:32"}},
 "warsage": {"video": "warsage/p1-warsage-2026-09-19.mp4", "score": "1-1", "buts": {"42:06": "A", "111:01": "B"},
             "tenues": (("green", ""), ("red", "")),                # 0-1 vert, 1-1 rouge
             "bornes": {"KO1": "02:25", "Fin1MT": "48:00", "KO2": "63:01", "FinMatch": "111:51"}},
 "baelen": {"video": "baelen/p4-baelen-2026-09-12.mp4", "score": "4-3",
            "buts": {"09:27": "A", "10:17": "A", "26:39": "A", "65:56": "B", "72:55": "B", "77:19": "B", "92:09": "A"},
            "tenues": (("light blue", ""), ("black", "red")),      # bleu clair ; maillot noir, short rouge
            "bornes": {"KO1": "06:38", "Fin1MT": "52:45", "KO2": "65:40", "FinMatch": "115:00"}},
}
_s_ = lambda h: int(h.split(":")[0]) * 60 + int(h.split(":")[1])
for _n, _d in NOUVEAUX.items():
    VERITE_EQUIPES[_n] = dict(_d["buts"])
    VRAIES_TENUES_TOUTES[_n] = _d["tenues"]
    _VID[_n] = _d["video"]
    _KO_CONNU[_n] = float(_s_(_d["bornes"]["KO1"]))
MATCHS = {m: {"video": f"{V_}/{v}", "camera_type": "high_side",
              "buts": sorted(VERITE_EQUIPES[m], key=lambda h: int(h.split(":")[0]) * 60 + int(h.split(":")[1])),
              "coup_envoi_s": _KO_CONNU.get(m), "ko2_s": None} for m, v in _VID.items()}
if MATCH not in MATCHS:
    raise SystemExit(f"❌ MATCH = {MATCH!r} inconnu — choisir parmi {sorted(MATCHS)}")
_M = MATCHS[MATCH]
LOG_FILE = f"/kaggle/working/outputs/{MATCH}_reference.log"
OUT_DIR = f"outputs/{MATCH}_reference"
GEMINI_CACHE = "on"
FILTRAGE_PISTES = ""         # suivi de PRODUCTION (historique)
KO_CONSIGNE, KO_FENETRE, KO_TEMPERATURE = "fenetre", "30", "0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s",
                    handlers=[logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")])
for noisy in ["PIL", "pytesseract", "ultralytics", "numba", "librosa", "absl", "torchreid", "matplotlib"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)
for _h in logging.getLogger().handlers:
    _h.addFilter(lambda _r: not _r.name.startswith("numba"))
log = logging.getLogger()
if hasattr(builtins.print, "__wrapped__"):
    builtins.print = builtins.print.__wrapped__
_orig_print = builtins.print
def _print(*args, **kwargs):
    log.info(" ".join(str(a) for a in args))
    _orig_print(*args, **kwargs)
_print.__wrapped__ = _orig_print
builtins.print = _print

# ── Clés : lues dans le fichier param.env du dataset PRIVÉ « mziparameter » (plus aucune clé en clair ici) ──
#    à défaut : variables d'environnement, puis secrets Kaggle. Les clés ne sont jamais affichées.
PARAM_ENV = "/kaggle/input/datasets/michaelziant/mziparameter/param.env"
_PARAMS = {}
_pe = PARAM_ENV if os.path.exists(PARAM_ENV) else next(iter(__import__("glob").glob("/kaggle/input/**/mziparameter/**/param.env", recursive=True)), None)
if _pe:
    for _l in open(_pe, encoding="utf-8"):
        _l = _l.strip()
        if _l and not _l.startswith("#") and "=" in _l:
            _k, _v = _l.split("=", 1)
            _PARAMS[_k.strip()] = _v.strip().strip('"').strip("'")
    print(f"  Clés lues dans {_pe} : {', '.join(k for k in _PARAMS if k.endswith('_KEY'))}")
else:
    print(f"  ⚠️ {PARAM_ENV} introuvable — ajoute le dataset « mziparameter » en entrée")
def _secret(nom):
    v = _PARAMS.get(nom, "") or os.getenv(nom, "")
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient
            v = UserSecretsClient().get_secret(nom) or ""
        except Exception:
            v = ""
    return v
GEMINI_KEY = _secret("GEMINI_API_KEY")
CLAUDE_KEY = _secret("CLAUDE_API_KEY")
if not GEMINI_KEY:
    raise SystemExit("❌ GEMINI_API_KEY introuvable (ni dans param.env de « mziparameter », ni dans les secrets) : run annulé")

print("\n" + "=" * 50 + "\nÉTAPE 1/3 — Clone\n" + "=" * 50)
os.chdir("/kaggle/working")
if os.path.exists("/kaggle/working/ProjectAIMika"):
    shutil.rmtree("/kaggle/working/ProjectAIMika")
if os.system("git clone --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git /kaggle/working/ProjectAIMika") != 0:
    raise SystemExit("❌ Clone échoué")
print("Clone OK")

print("\n" + "=" * 50 + "\nÉTAPE 2/3 — Dépendances\n" + "=" * 50)
os.system("pip install -q av torchreid flask flask-sqlalchemy flask-login opencv-python-headless anthropic fpdf2 stripe "
          "python-dotenv werkzeug deep-sort-realtime scikit-learn google-genai joblib numpy psutil")
os.system("apt-get install -q tesseract-ocr ffmpeg")
os.system("pip install -q pytesseract")
print("Dépendances OK")

print("\n" + "=" * 50 + "\nÉTAPE 3/3 — Environnement\n" + "=" * 50)
os.chdir("/kaggle/working/ProjectAIMika")
sys.path.insert(0, "/kaggle/working/ProjectAIMika")
for d in ["uploads", "outputs", "instance", "models", ".env", "outputs/learning", "outputs/learning/football", OUT_DIR, OUT_DIR + "/cache"]:
    os.makedirs(d, exist_ok=True)
for pkg in ["analysis", "analytics", "ai", "vision", "video", "rendering", "export", "payments", "sports", "tracking"]:
    os.makedirs(pkg, exist_ok=True)
    if not os.path.exists(f"{pkg}/__init__.py"):
        open(f"{pkg}/__init__.py", "w").close()
with open(".env/param.env", "w") as f:
    f.write(f"""SECRET_KEY=test-kaggle
DATABASE_URI=sqlite:///db.sqlite3
DEBUG=false
UPLOAD_FOLDER=uploads
OUTPUT_FOLDER=outputs
FPS=25
FRAME_WIDTH=1920
FRAME_HEIGHT=1080
YOLO_MODEL=yolo26m.pt
YOLO_CONFIDENCE=0.4
BALL_METHOD=hybrid
CLAUDE_API_KEY={CLAUDE_KEY}
CLAUDE_MODEL=claude-sonnet-4-5
GEMINI_API_KEY={GEMINI_KEY}
R2_ENABLED=false
YOLO_BATCH_SIZE=4
YOLO_IMGSZ=640
FRAME_SKIP_EVERY=4
FILE_RETENTION_DAYS=30
""")
os.environ["GEMINI_API_KEY"] = GEMINI_KEY
os.environ["CLAUDE_API_KEY"] = CLAUDE_KEY
os.environ["DISABLE_OCR"] = "1"
os.environ["GEMINI_MAX_PARALLEL"] = "128"   # 13/11 : questions Gemini « d'un coup » (validé sur 3 matchs : mêmes buts, 0 attribution fausse, pipeline replay 11 à 15 min)
os.environ.pop("TRACKER_MAX_PREDICTION_AGE", None)
os.environ["SCOUTIA_TRACKER_MODE"] = "historique"
os.environ["SCOUTIA_KO_CONSIGNE"] = KO_CONSIGNE
os.environ["SCOUTIA_KO_FENETRE"] = KO_FENETRE
os.environ["SCOUTIA_KO_TEMPERATURE"] = KO_TEMPERATURE
print("Environnement OK")


# ────────────────────────────────────────────────────────────

print("\n" + "=" * 50 + "\nCONTRÔLE PRÉ-VOL\n" + "=" * 50)
_R = "/kaggle/working/ProjectAIMika"
_src = open(f"{_R}/pipeline.py", encoding="utf-8").read()
_src_gv = open(f"{_R}/ai/gemini_validator.py", encoding="utf-8").read()
_src_main = open(f"{_R}/main.py", encoding="utf-8").read()
_src_pdf = open(f"{_R}/export/pdf.py", encoding="utf-8").read()
_checks = {
    "KO start_s explicite (replay OK)": "_ko_start_s = max(0.0, _kickoff_offset - 10.0)" in _src,
    "_ko_goals compare aux buts seulement": 'for g in _goals_deduped\n                if g.get("type") in ("goal", "score")' in _src,
    "pipeline : statut des buts + KO2 + temps forts": "_ko2_cand_idx" in _src and '"temps_forts":' in _src,
    "pipeline : contrôle final des doublons": "_temps_forts_final" in _src,
    "pipeline : mi-temps en parallèle (bornes)": "def _analyser_mi_temps(" in _src and "_DECALAGE_IDS_MT2" in _src,
    "pipeline : règle de la pause SÛRE (28/10)": "_MARGE_PAUSE_MI_TEMPS_S" in _src,
    "pipeline : voie « ballon » désactivée par défaut": '"SCOUTIA_VOIE_BALLON", "0"' in _src,
    "pipeline : second regard activé par défaut": '"SCOUTIA_KO_SECOND_REGARD", "1"' in _src,
    "pipeline : buts avec statut + identification": '"identification_joueurs":' in _src and "goals_a_verifier" in _src,
    "pipeline : doublons de temps forts quelle que soit l'issue (31/10)": "QUEL QUE SOIT le résultat vu par Gemini" in _src,
    "export/pdf : frise des Top Highlights sur plusieurs lignes (31/10)": "_fin_ligne" in _src_pdf,
    "gemini_validator : réflexion réduite (KO, classification)": "def _appliquer_reflexion_reduite(" in _src_gv,
    "gemini_validator : modèle unique + cache pour tous": "_installer_verrou_gemini" in _src_gv,
    "gemini_validator : cache par match": "SCOUTIA_GEMINI_CACHE_VIDEO" in _src_gv,
    "player_reid : version VALIDÉE": "linear_sum_assignment" not in open(f"{_R}/analysis/player_reid.py", encoding="utf-8").read(),
    "replay : le cache conserve time_since_update": '"time_since_update": p.get("time_since_update")' in open(f"{_R}/replay.py", encoding="utf-8").read(),
}
_ok = True
for _k, _v in _checks.items():
    print(f"  {'✅' if _v else '❌'} {_k}")
    _ok = _ok and _v
import glob as _g_prod
_att_mod = _g_prod.glob("/kaggle/working/ProjectAIMika/**/attribution_equipes.py", recursive=True)
_att_src = open(_att_mod[0], encoding="utf-8").read() if _att_mod else ""
for _nom, _v in {"ai/attribution_equipes.py présent (11/11 : normale + limite 30 Mo)": "requetes_recompressees" in _att_src,
                 "pipeline : attribution des équipes (Step 11b)": "Step 11b" in open(
                     _g_prod.glob("/kaggle/working/ProjectAIMika/**/pipeline.py", recursive=True)[0], encoding="utf-8").read()}.items():
    print(f"  {'✅' if _v else '❌'} {_nom}")
    _ok = _ok and _v
if not _ok:
    raise SystemExit("❌ La version sur GitHub n'est PAS la bonne — run annulé")
import torch as _torch
if _torch.cuda.is_available():
    print(f"  GPU : {_torch.cuda.get_device_name(0)}\n✅ Version correcte — lancement")
else:
    # 17/11 : sans GPU, chaque processus de mi-temps prend la moitié des cœurs (au lieu d'un seul fil)
    _coeurs = os.cpu_count() or 2
    for _v_f in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
        os.environ[_v_f] = str(max(1, _coeurs // 2))
    print(f"  ⚠️ PAS DE GPU : analyse sur processeur ({_coeurs} cœurs, {max(1, _coeurs // 2)} fils par mi-temps) — "
          f"BEAUCOUP plus lent ; une session Kaggle s'arrête au bout de 12 h (à vérifier)\n✅ Lancement")


# ────────────────────────────────────────────────────────────

VIDEO = _M["video"]
if not os.path.exists(VIDEO):
    raise SystemExit(f"❌ Vidéo introuvable : {VIDEO}")
os.environ["SCOUTIA_GEMINI_CACHE"] = GEMINI_CACHE
os.environ["SCOUTIA_STOP_APRES_SUIVI"] = ""
os.environ["SCOUTIA_KO_DEDUP"] = KO_DEDUP
import json as _json_eq
os.environ["SCOUTIA_EQUIPES"] = _json_eq.dumps(EQUIPES, ensure_ascii=False)
os.environ["SCOUTIA_PROFIL_KO"] = "2026-11"     # 17/11 : configuration validée (prime sur KO_DEDUP ci-dessus)
os.environ["SCOUTIA_KO_SECOND_REGARD"] = "1" if SECOND_REGARD else "0"
os.environ["SCOUTIA_TF_DESCRIPTIONS_KO"] = "1" if OCCASIONS_KO else ""
os.environ["SCOUTIA_VOIE_BALLON"] = "1" if VOIE_BALLON else "0"
os.environ["SCOUTIA_REFLEXION_REDUITE"] = "1" if REFLEXION_REDUITE else "0"
os.environ["SCOUTIA_LECTURE_MAILLOTS"] = "1" if LECTURE_MAILLOTS else "0"
os.environ["SCOUTIA_GEMINI_CACHE_OUT"] = f"/kaggle/working/gemini_cache_{MATCH}.jsonl"
os.environ["SCOUTIA_GEMINI_CACHE_IN"] = "/kaggle/working/aucun_cache_lu.jsonl"   # rien n'est relu → VRAI PRIX
os.environ["SCOUTIA_GEMINI_CACHE_VIDEO"] = VIDEO
import sys as _sys_purge
_racines = ("pipeline", "main", "config", "replay", "ai", "analysis", "vision", "rendering", "utils", "segment_extractor")
for _m in [m for m in list(_sys_purge.modules) if m.split(".")[0] in _racines]:
    del _sys_purge.modules[_m]
# ── 15/11 : DÉTECTEUR LIBRE RF-DETR à la place de YOLO (en mémoire, aucun fichier modifié) ──
DETECTEUR_TAILLE = "medium"
import subprocess as _sp_rf
_sp_rf.run([sys.executable, "-m", "pip", "install", "-q", "rfdetr"], check=False)
# 15/11 : Ultralytics DÉSINSTALLÉ pour ce run : si un module du pipeline en dépend encore, le run s'arrête sur une
# erreur « No module named 'ultralytics' » (c'est le but : prouver que le pipeline tourne sans lui).
_sp_rf.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "ultralytics"], check=False)
for _m_u in [m for m in list(sys.modules) if m.split(".")[0] == "ultralytics"]:
    del sys.modules[_m_u]
os.environ["SCOUTIA_DETECTEUR"] = "rfdetr"
os.environ["SCOUTIA_RFDETR_TAILLE"] = DETECTEUR_TAILLE
try:
    import vision.detecteur_libre as _dl
except ImportError:
    raise SystemExit("❌ vision/detecteur_libre.py absent du dépôt GitHub — rien n'est lancé")
if not _dl.brancher():
    raise SystemExit("❌ RF-DETR non branché — rien n'est lancé")
if not os.path.exists("/kaggle/working/ProjectAIMika/analysis/profil_ko.py"):
    raise SystemExit("❌ analysis/profil_ko.py absent de GitHub (fichiers du 17/11) — rien n'est lancé")
from pipeline import run_pipeline
print("\n" + "=" * 50 + f"\nANALYSE POUR LE JOUEUR — {MATCH} (suivi + bornes + buts)\n" + "=" * 50)
pipeline_t0 = time.time()
result = {}
try:
    result = run_pipeline(video_path=VIDEO, sport="football", output_dir=OUT_DIR, camera_type=_M["camera_type"],
                          use_single_yolo26=True, ball_model_name="yolo26m.pt", ball_conf=0.15, video_end_s=None,
                          kickoff_s_precalcule=None, use_match_boundary_segments=SEGMENTS, _match_data=None)
finally:
    print(f"\nPipeline runtime : {(time.time() - pipeline_t0) / 60:.1f} min")
    _cache_dir = f"/kaggle/working/ProjectAIMika/{OUT_DIR}/cache"
    if os.path.exists(os.path.join(_cache_dir, "meta.json")):
        shutil.make_archive(f"/kaggle/working/cache_{MATCH}_reference", "zip", _cache_dir)
        print(f"✅ Cache de suivi sauvegardé : /kaggle/working/cache_{MATCH}_reference.zip "
              f"({os.path.getsize(f'/kaggle/working/cache_{MATCH}_reference.zip') / 1e6:.0f} MB)")
    else:
        print("⚠️  Pas de cache de suivi à sauvegarder")


# ────────────────────────────────────────────────────────────

# ── Comparaison des BUTS avec la vérité (détection seule) + aperçu du rendu client ──
_VRAIS = [int(b.split(":")[0]) * 60 + int(b.split(":")[1]) for b in _M["buts"]]
_KO_DET = float((result or {}).get("kickoff_offset") or 0.0)
_fm = lambda t: f"{int(t)//60:02d}:{int(t)%60:02d}"
_ann = [(_g.get("time", 0) + _KO_DET, _g.get("goal_status", "—")) for _g in (result or {}).get("events", []) if _g.get("type") in ("goal", "score")]
print("\n" + "=" * 60 + "\nCOMPARAISON AVEC LES BUTS RÉELS\n" + "=" * 60)
print(f"  Coup d'envoi retenu : {_fm(_KO_DET)} (connu : {_fm(_M['coup_envoi_s']) if _M['coup_envoi_s'] else 'non noté'})")
for _v in _VRAIS:
    _m = [(t, st) for t, st in _ann if abs(t - _v) <= 25]
    print(f"  but réel {_fm(_v)} : " + (f"✅ trouvé à {_fm(_m[0][0])} — {_m[0][1]}" if _m else "❌ MANQUÉ"))
_fp = [(t, st) for t, st in _ann if not any(abs(t - _v) <= 25 for _v in _VRAIS)]
for t, st in _fp:
    print(f"  but annoncé {_fm(t)} sans but réel — {st}")
_bm = (result or {}).get("bornes_mi_temps")
if _bm and _bm.get("segments"):
    print(f"  Bornes : KO1 {_fm(_bm['ko1'])} | Fin1MT {_fm(_bm['fin1mt'])} | KO2 {_fm(_bm['ko2'])} | FinMatch {_fm(_bm['finmatch'])}")
    if MATCH in NOUVEAUX:
        _bn = NOUVEAUX[MATCH]["bornes"]
        print(f"  Bornes notées : KO1 {_bn['KO1']} | Fin1MT {_bn['Fin1MT']} | KO2 {_bn['KO2']} | FinMatch {_bn['FinMatch']}")
_tp = sum(1 for _v in _VRAIS if any(abs(t - _v) <= 25 for t, _ in _ann))
print(f"  BILAN DÉTECTION : {_tp}/{len(_VRAIS)} vrais buts | {len(_fp)} faux positifs dont {sum(st == 'confirmé' for _, st in _fp)} « confirmé(s) »")
try:
    from export.pdf import generate_pdf as _gen_pdf
    _gen_pdf(result, f"/kaggle/working/apercu_{MATCH}.pdf", sport="football")
    print(f"  Aperçu PDF : /kaggle/working/apercu_{MATCH}.pdf")
except Exception as _e_ap:
    print(f"  ⚠️ Aperçu PDF impossible : {type(_e_ap).__name__} : {_e_ap}")
try:
    import ai.gemini_validator as _gv_cache
    _gv_cache.gemini_cache_summary()
except Exception as _e_gc:
    print(f"  [GEMINI CACHE] bilan indisponible : {_e_gc}")


# ────────────────────────────────────────────────────────────


# ── relais vers la cellule 2/2 ──
import json as _json_be
_bm = (result or {}).get("bornes_mi_temps") or {}
_bornes = None
if all(_bm.get(k) is not None for k in ("ko1", "fin1mt", "ko2", "finmatch")):
    _bornes = [[float(_bm["ko1"]), float(_bm["fin1mt"])], [float(_bm["ko2"]), float(_bm["finmatch"])]]
_buts = [{"time_abs": float(b["time_abs"]), "statut": b.get("status") or b.get("goal_status") or b.get("statut") or "?",
          "equipe": b.get("tenue_equipe"), "niveau_equipe": b.get("niveau_equipe")}
         for b in (result or {}).get("buts", []) if b.get("time_abs") is not None]
if not _buts:   # repli : les événements « goal » du résultat
    _buts = [{"time_abs": float(g.get("time", 0)) + _KO_DET, "statut": g.get("goal_status", "?")}
             for g in (result or {}).get("events", []) if g.get("type") in ("goal", "score")]
_cache_be = f"/kaggle/working/ProjectAIMika/{OUT_DIR}/cache"
_json_be.dump({"match": MATCH, "video": VIDEO, "cache": _cache_be, "actions_abs": [], "bornes": _bornes, "buts": _buts,
               "t0": T0_BOUT_EN_BOUT, "duree_analyse_min": round((time.time() - T0_BOUT_EN_BOUT) / 60, 1), "joueur_seul": True},
              open(f"/kaggle/working/bout_en_bout_{MATCH}.json", "w"))
print(f"\n══ JOUEUR SEUL, ÉTAPE 1 TERMINÉE : {(time.time() - T0_BOUT_EN_BOUT) / 60:.0f} min | bornes : "
      + (" ; ".join(f"{_fm(a)}-{_fm(b)}" for a, b in _bornes) if _bornes else "NON TROUVÉES (tout le fichier sera lu)")
      + f" | {len(_buts)} but(s) : " + ", ".join(f"{_fm(b['time_abs'])} ({b['statut']})" for b in _buts))

# ── PROD : attribution, livrables et COÛT RÉEL de l'analyse (réponses Gemini écrites pendant ce run) ──
import json as _jp, collections as _cp
DUREE_ANALYSE_MIN = (time.time() - T0_BOUT_EN_BOUT) / 60
_att = (result or {}).get("attribution_equipes") or {}
if _att:
    print(f"   ÉQUIPES : {_att.get('tenues')} | score confirmé {_att.get('score_confirme')} | estimé {_att.get('score_estime')} | "
          f"sans équipe {_att.get('buts_sans_equipe')} | recompressées {_att.get('requetes_recompressees')} | "
          f"échecs {_att.get('questions_en_echec')}")
else:
    print("   ⚠️ ÉQUIPES : aucune attribution dans le résultat (module absent ou erreur : voir « [ATTRIBUTION] »)")
_rapport = f"/kaggle/working/ProjectAIMika/{OUT_DIR}/rapport.pdf"
if os.path.exists(_rapport):
    shutil.copy(_rapport, f"/kaggle/working/rapport_{MATCH}.pdf")
    print(f"   RAPPORT CLIENT : /kaggle/working/rapport_{MATCH}.pdf")
COUT_ANALYSE = _cp.defaultdict(lambda: [0, 0.0])
_out = os.environ.get("SCOUTIA_GEMINI_CACHE_OUT")
if _out and os.path.exists(_out):
    for _l in open(_out, encoding="utf-8"):
        try:
            _x = _jp.loads(_l)
            _ts = _dt_prod.datetime.fromisoformat(str(_x.get("enregistre_le") or "")[:19]).timestamp()
        except Exception:
            continue
        if _ts < _T_DEBUT_GEMINI - 120:
            continue
        _u = _x.get("usage") or {}
        _c = (int(_u.get("prompt_token_count") or 0) / 1e6 * 0.75
              + (int(_u.get("candidates_token_count") or 0) + int(_u.get("thoughts_token_count") or 0)) / 1e6 * 3.75)
        _f = (_x.get("contexte") or {}).get("fonction", "?")
        COUT_ANALYSE[_f][0] += 1; COUT_ANALYSE[_f][1] += _c
print(f"\n══ COÛT GEMINI DE L'ANALYSE ══")
for _f, (_n, _c) in sorted(COUT_ANALYSE.items(), key=lambda kv: -kv[1][1]):
    print(f"   {_f:34s} {_n:4d} question(s) ≈ {_c:.3f} $")
print(f"   TOTAL analyse ≈ {sum(v[1] for v in COUT_ANALYSE.values()):.2f} $ ({sum(v[0] for v in COUT_ANALYSE.values())} questions)")
# ── 15/11 : contrôle automatique des ÉQUIPES contre la vérité terrain (même règle que les tests de replay) ──
import re as _re_v, collections as _col_v
_att_v = (result or {}).get("attribution_equipes") or {}
_ten_v = _att_v.get("tenues") or {}
def _parse_v(t):
    m = _re_v.match(r"maillot (.*) / short (.*)", str(t or ""))
    return (m.group(1), m.group(2)) if m else ("", "")
_TD_v = (_parse_v(_ten_v.get("A")), _parse_v(_ten_v.get("B")))
_mots_v = lambda x: set(str(x).lower().replace(",", " ").replace("/", " ").split()) - {"and", "with", "dark", "light"}
_sc_v = lambda det, vrai: len(_mots_v(det[0]) & _mots_v(vrai[0])) * 2 + len(_mots_v(det[1]) & _mots_v(vrai[1]))
_VT_v = VRAIES_TENUES_TOUTES[MATCH]; _VE_v = VERITE_EQUIPES[MATCH]
if _VT_v is None:      # 17/11 : nouveaux matchs — équipes des buts non notées : attribution affichée, pas comparée
    _map_v = None
else:
    _d_v = _sc_v(_TD_v[0], _VT_v[0]) + _sc_v(_TD_v[1], _VT_v[1]); _i_v = _sc_v(_TD_v[0], _VT_v[1]) + _sc_v(_TD_v[1], _VT_v[0])
    _map_v = None if _d_v == _i_v else ({"A": "A", "B": "B"} if _d_v > _i_v else {"A": "B", "B": "A"})
_bil_v = _col_v.Counter(); _vu_v = set()
for _b_v in (result or {}).get("buts", []):
    _g_v = float(_b_v.get("time_abs") or 0)
    _m_v = [h for h in _VE_v if abs(int(h.split(":")[0]) * 60 + int(h.split(":")[1]) - _g_v) <= 25 and h not in _vu_v]
    if not _m_v:
        continue
    _vu_v.add(_m_v[0])
    _e_v = _b_v.get("equipe")
    _ok_v = "—" if not _e_v else ("?" if not _map_v else ("juste" if _map_v[_e_v] == _VE_v[_m_v[0]] else "FAUX"))
    _bil_v[(_b_v.get("niveau_equipe") or "non attribué", _ok_v)] += 1
_n_trouves = sum(1 for _v in _VRAIS if any(abs(t - _v) <= 25 for t, _ in _ann))
print(f"\n══ BILAN RF-DETR — {MATCH} ══")
print(f"   BUTS : {_n_trouves}/{len(_VRAIS)} vrais buts | faux buts {len(_fp)} dont « confirmé » "
      f"{sum(1 for _, st in _fp if str(st).startswith('confirm'))}")
print(f"   ÉQUIPES : tenues {'OK' if _map_v else 'CORRESPONDANCE IMPOSSIBLE'} | "
      f"justes {_bil_v[('confirmé', 'juste')] + _bil_v[('probable', 'juste')]} | "
      f"FAUX {sum(v for k, v in _bil_v.items() if k[1] == 'FAUX')} | "
      f"non attribués {sum(v for k, v in _bil_v.items() if k[0] == 'non attribué')}")
print(f"   DURÉE de l'analyse {(time.time() - T0_BOUT_EN_BOUT) / 60:.1f} min | Gemini ≈ {sum(v[1] for v in COUT_ANALYSE.values()):.2f} $")

# ── 21/11 : VIDÉO RÉSUMÉ DU MATCH (tous les buts + toutes les occasions), produite par le pipeline ──
_reel = (result or {}).get("reel")
_reel = os.path.join("/kaggle/working/ProjectAIMika", _reel) if _reel and not os.path.isabs(_reel) else _reel
if _reel and os.path.exists(_reel):
    shutil.copy(_reel, f"/kaggle/working/resume_{MATCH}.mp4")
    print(f"\n✅ RÉSUMÉ DU MATCH : /kaggle/working/resume_{MATCH}.mp4")
else:
    print("\n⚠️ Pas de vidéo résumé du match (vérifier export/resume_match.py sur GitHub)")

# ════ ÉTAPE 2 — JOUEURS : module de production analysis/montage_joueur.py (comme le fera le pipeline) ════
if not JOUEURS:
    print("\n   Aucun joueur demandé (JOUEURS vide) — fin du run après l'analyse.")
else:
    os.system("pip install -q pypdf timm pytorch-lightning nltk")   # fiche jointe au rapport + lecteur de numéros PARSeq
    sys.path.insert(0, "/kaggle/working/ProjectAIMika")
    from analysis.montage_joueur import monter_joueur
    _rap_mj = (result or {}).get("pdf")
    _rap_mj = os.path.join("/kaggle/working/ProjectAIMika", _rap_mj) if _rap_mj and not os.path.isabs(_rap_mj) else _rap_mj
    for _j in JOUEURS:
        _sortie_j = f"/kaggle/working/joueur_{_j['numero']}_{_j['couleur'].replace(' ', '_')}"
        print(f"\n{'═' * 90}\nJOUEUR n°{_j['numero']} {_j['couleur']}{' (gardien)' if _j.get('gardien') else ''}\n{'═' * 90}", flush=True)
        _res_j = monter_joueur(video=VIDEO, cache_dir=_cache_be, sortie=_sortie_j, match=MATCH, bornes=_bornes, buts=_buts,
                               entrees=["/kaggle/input", "/kaggle/working"], gemini_key=GEMINI_KEY,
                               rapport_pdf=_rap_mj if _rap_mj and os.path.exists(_rap_mj) else None, **_j)
        for _k in ("trouve", "erreur", "message", "presence_s", "passages", "moments_ballon", "buts_confirmes",
                   "buts_possibles", "arrets", "tirs_adverses", "cout_gemini_usd", "duree_min", "montage", "fiche",
                   "rapport_et_fiche"):
            if _k in _res_j:
                print(f"   {_k:18s} : {_res_j[_k]}")
print("\nÀ garder : cache_<match>_reference.zip, bout_en_bout_<match>.json, gemini_cache_<match>.jsonl, rapports, dossiers joueur_*.")

  Clés lues dans /kaggle/input/datasets/michaelziant/mziparameter/param.env : SECRET_KEY, CLAUDE_API_KEY, GEMINI_API_KEY, STRIPE_SECRET_KEY, STRIPE_PUBLIC_KEY, R2_SECRET_ACCESS_KEY, RUNPOD_API_KEY

ÉTAPE 1/3 — Clone


Cloning into '/kaggle/working/ProjectAIMika'...


Clone OK

ÉTAPE 2/3 — Dépendances
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.6/360.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
Dépendances OK

ÉTAPE 3/3 — Environnement
Environnement OK

CONTRÔLE PRÉ-VOL
  ✅ KO start_s explicite (replay OK)
  ✅ _ko_goals compare aux buts seulement
  ✅ pipeline : statut des buts + KO2 + temps forts
  ✅ pipeline : contrôle final des doublons
  ✅ pipeline : mi-temps en parallèle (bornes)
  ✅ pipeline : règle de la pause SÛRE (28/10)
  ✅ pipeline : voie « ballon » désactivée par défaut
  ✅ pipeline : second regard activé par défaut
  ✅ pipeline : buts avec statut + identification
  ✅ pipeline : doublons de temps forts quelle que soit l'issue (31/10)
  ✅ export/pdf : frise des Top Highlights sur plusieurs lignes (31/10)
  ✅ gemini_validator : réflexion réduite (KO, classification)
  ✅ gemini_validator : modèle u

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.3 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 2.4.0 which is incompatible.


... (11 outputs left out)

[2026-11-23 14:33:50] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. Create predictor = native_model.inference(...) and use predictor.predict(...). For a checkpoint, first load native_model = RFDETR.from_checkpoint(...).


  ✅ goal CONFIRMÉ (HIGH_SIDE_NO_SHOT) à t=6464.1s — vitesse=False trajectoire=True goal_cd=0 x=1792(93.3%) y=549(50.8%) lost_frames=0 pas_contradictoire=True
  [RF-DETR processus 2173] 16000 images | 21.6 joueurs/image | ballon sur 64 % des images | optimisé : oui


[2026-11-23 14:54:18] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. Create predictor = native_model.inference(...) and use predictor.predict(...). For a checkpoint, first load native_model = RFDETR.from_checkpoint(...).


  ✅ goal CONFIRMÉ (HIGH_SIDE_NO_SHOT) à t=6893.9s — vitesse=True trajectoire=False goal_cd=0 x=0(0.0%) y=905(83.8%) lost_frames=1 pas_contradictoire=True
  [END_TIME_S] Arrêt de la lecture à frame 209850 (limite atteinte)

  209850 frames lues | 18527 analysées | batches de 4
  [MAILLOTS] lecture des maillots par Gemini désactivée (SCOUTIA_LECTURE_MAILLOTS=0)
  [GEMINI JERSEYS] échec, repli intégral sur Tesseract : lecture des maillots désactivée
  24458 events detectes
  0 maillots identifies
  [DIAG team gap] n=788685  %<15.0=100.0%  percentiles(10/25/50/75/90)=[0.3, 0.6, 0.8, 1.2, 1.5]

  [DIAG CALIBRATION+CLASSIFICATION]
  Calibration
  -----------
  samples utilisés       : 399
  cluster 0              : 160
  cluster 1              : 239
  ratio                  : 40.1% / 59.9%
  distance centroides    : 1.69

  Classification (sur 788844 decisions post-calibration)
  --------------
  team 0                 : 40.0%  (315689)
  team 1                 : 48.4%  (381510)
  unknown (a